# Data Fusion 2025 — Задача 2 "4cast"
## Прогнозирование еженедельных переводов юридических лиц

**Задача:** предсказать суммарные переводы 51 963 клиентов банка за 12 недель (118–129),  
используя историю за 118 недель (0–117).

**Метрика:** средний по клиентам RMSLE:
$$\overline{RMSLE} = \frac{1}{N}\sum_{i=1}^N \sqrt{\frac{1}{T}\sum_{t=1}^T (\log(1+y_{it}) - \log(1+\hat{y}_{it}))^2}$$

**Public leaderboard:** недели 118–121 (4 из 12 прогнозных).

---


## Скачивание и предобработка данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import lightgbm as lgb
import warnings
import json
from pathlib import Path
import gc, time

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.4
sns.set_palette("tab10")

DATA_DIR = Path("data")
FIG_DIR  = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# Splits (из calendar_extended.csv)
TRAIN_WEEKS    = list(range(0, 106))     # 0–105  → train
VAL_PUB_WEEKS  = list(range(106, 110))  # 106–109 → validation_public
VAL_PRIV_WEEKS = list(range(110, 118))  # 110–117 → validation_private
PUBLIC_WEEKS   = list(range(118, 122))  # 118–121 → public leaderboard
PRIVATE_WEEKS  = list(range(122, 130))  # 122–129 → private leaderboard
FORECAST_WEEKS = list(range(118, 130))  # все 12 недель прогноза

print("Config loaded.")


Config loaded.


In [2]:
print("Loading data...")
ts_train = pd.read_parquet(DATA_DIR / "target_series.parquet")       # weeks 0–105
ts_ext   = pd.read_parquet(DATA_DIR / "target_series_extended.parquet")  # weeks 106–117
calendar = pd.read_csv(DATA_DIR / "calendar_extended.csv", parse_dates=["date"])
sample   = pd.read_csv(DATA_DIR / "sample_submit_extended.csv")

# Объединяем все известные таргеты (недели 0–117)
df_all = pd.concat([ts_train, ts_ext], ignore_index=True)
df_all["week"] = df_all["week"].astype(int)

TX_FILES = sorted(DATA_DIR.glob("transactions_[1-5].parquet"))
print(f"Found {len(TX_FILES)} transaction files: {[f.name for f in TX_FILES]}")

t0 = time.time()
trns_chunks = []
for f in TX_FILES:
    print(f"Reading {f.name} ...")
    chunk = pd.read_parquet(f)
    print(f"  -> {len(chunk):,} rows")
    trns_chunks.append(chunk)

trns_raw = pd.concat(trns_chunks, ignore_index=True)
del trns_chunks
gc.collect()

Loading data...
Found 5 transaction files: ['transactions_1.parquet', 'transactions_2.parquet', 'transactions_3.parquet', 'transactions_4.parquet', 'transactions_5.parquet']
Reading transactions_1.parquet ...
  -> 52,089,892 rows
Reading transactions_2.parquet ...
  -> 55,045,806 rows
Reading transactions_3.parquet ...
  -> 59,510,350 rows
Reading transactions_4.parquet ...
  -> 61,492,247 rows
Reading transactions_5.parquet ...
  -> 30,036,951 rows


0

## Подготовка данных 

Идея: собрать для каждого инн временной ряд с суммарными переводами за каждый день со счетов втб в другие банки и другим счетам инн

In [3]:
trns = trns_raw.copy()

In [4]:
trns["date"] = pd.to_datetime(trns["date"], utc=True).dt.tz_localize(None).dt.normalize()

all_inn = pd.read_parquet(DATA_DIR / "target_series.parquet", columns=["inn_id"])["inn_id"].unique()

cal_117 = calendar.loc[calendar["week"] <= 117].copy()
cal_117["date"] = pd.to_datetime(cal_117["date"]).dt.tz_localize(None).dt.normalize()
all_dates = np.sort(cal_117["date"].drop_duplicates().values)

mi = pd.MultiIndex.from_product([all_inn, all_dates], names=["doc_payer_inn", "date"])
new_trns = mi.to_frame(index=False)

n = len(new_trns)
new_trns["doc_payee_inn"] = ""
new_trns["trns_count"] = 1.0
new_trns["trns_amount"] = 0
new_trns["doc_payer_bank_name_encoded"] = 51
new_trns["doc_payee_bank_name_encoded"] = 0
new_trns["doc_payer_bank_name_flag"] = 1
new_trns["doc_payee_bank_name_flag"] = 0
new_trns["trns_class_encoded"] = 0

trns = pd.concat([trns, new_trns], ignore_index=True)

In [5]:
trns.head()

,date,doc_payer_inn,doc_payee_inn,trns_count,trns_amount,doc_payer_bank_name_encoded,doc_payee_bank_name_encoded,doc_payer_bank_name_flag,doc_payee_bank_name_flag,trns_class_encoded
0,2022-07-25,inn1000006,inn1444971,1.0,34293.820312,1165,51,0,1,45
1,2022-07-25,inn1000007,inn3671152,1.0,65274.714844,51,3343,1,0,45
2,2022-07-25,inn1000023,inn654268,1.0,14695.500977,4967,51,0,1,45
3,2022-07-25,inn1000031,inn5219166,1.0,13911.931641,2048,51,0,1,45
4,2022-07-25,inn1000051,inn5082671,1.0,380510.625000,51,51,1,1,38


In [6]:
trns = trns[trns["doc_payer_inn"] != trns["doc_payee_inn"]]
trns = trns[trns["doc_payer_bank_name_flag"] != 0]
trns = trns.reset_index(drop=True)
gc.collect()

0

In [7]:
trns = pd.merge(trns, calendar, on='date', how='left')

trns = trns.groupby(["doc_payer_inn", "date"], as_index=False, sort=False).agg({
    "trns_amount": "sum",
    "week": "first",
    "part": "first",
}).rename(columns={
                 "doc_payer_inn": "inn_id",
                 "trns_amount": "target_daily",
             })

idx = pd.MultiIndex.from_product([all_inn, all_dates], names=["inn_id", "date"])

df_daily = (
        trns.set_index(["inn_id", "date"])
             .reindex(idx, fill_value=0)
             .reset_index()
    )

In [8]:
df_daily.head()

,inn_id,date,target_daily,week,part
0,inn1000051,2022-07-25,380510.625000,0,train
1,inn1000051,2022-07-26,41648.703125,0,train
2,inn1000051,2022-07-27,0.000000,0,train
3,inn1000051,2022-07-28,0.000000,0,train
4,inn1000051,2022-07-29,0.000000,0,train


## Обучение

In [21]:
!SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True

In [ ]:
!pip3 install catboost


zsh:1: /Users/aeseverenkova/Desktop/bank_forecasting/venv/bin/pip3: bad interpreter: /Users/aeseverenkova/venv/bin/python3: no such file or directory
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
zsh:1: /Users/aeseverenkova/Desktop/bank_forecasting/venv/bin/pip3: bad interpreter: /Users/aeseverenkova/venv/bin/python3: no such file or directory
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python3.10 -m pip install --upgrade pip


In [31]:
!which pip3

/Users/aeseverenkova/Desktop/bank_forecasting/venv/bin/pip3


In [32]:
!pip3 install scikit-learn


[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [33]:
import numpy as np
from sklearn import metrics
import pandas as pd
from catboost import CatBoostRegressor

In [36]:
df = df_daily.sort_values("date").copy()

df["lag_1"] = df["target_daily"].shift(1)
df["lag_7"] = df["target_daily"].shift(7)
df["lag_14"] = df["target_daily"].shift(14)
df["lag_28"] = df["target_daily"].shift(28)

df["roll_mean_7"] = df["target_daily"].shift(1).rolling(7).mean()
df["dow"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df['weekend'] = df['dow'].isin([5,6])

data = df.dropna()
X = data[["lag_1", "lag_7", "lag_14", "lag_28", "roll_mean_7", "dow", "month", "weekend"]]
y = data["target_daily"]

split = df_daily[df_daily['week'] <= 109].tail(1).index[0]
X_train, X_valid = X.iloc[:split], X.iloc[split:]
y_train, y_valid = y.iloc[:split], y.iloc[split:]

y_train_log = np.log1p(y_train)

model = CatBoostRegressor(
    loss_function="RMSE",
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    random_seed=42,
    verbose=False
)

model.fit(X_train, y_train_log)
pred_log = model.predict(X_valid)
pred = np.expm1(pred_log)

In [37]:
pred

array([1.4289514, 1.4289514, 1.4289514, 1.4289514, 1.4289514, 1.4289514,
       1.4289514, 1.4289514, 1.4289514, 1.4289514, 1.4289514, 1.4289514,
       1.4289514, 1.4289514, 1.4289514, 1.4289514, 1.4289514, 1.4289514,
       1.4289514, 1.4289514, 1.4289514, 1.4289514, 1.4289514, 1.4289514,
       1.4289514, 1.4289514, 1.4289514, 1.4289514, 1.4289514])

In [40]:
df_daily[df_daily['week'] >= 109]

,inn_id,date,target_daily,week,part
763,inn1000051,2024-08-26,0.000000e+00,109,validation_public
764,inn1000051,2024-08-27,0.000000e+00,109,validation_public
765,inn1000051,2024-08-28,2.508962e+06,109,validation_public
766,inn1000051,2024-08-29,0.000000e+00,109,validation_public
767,inn1000051,2024-08-30,0.000000e+00,109,validation_public
...,...,...,...,...,...
42921433,inn999886,2024-10-23,3.375033e+06,117,validation_private
42921434,inn999886,2024-10-24,1.679644e+06,117,validation_private
42921435,inn999886,2024-10-25,1.228358e+06,117,validation_private
42921436,inn999886,2024-10-26,3.989773e+03,117,validation_private


In [41]:
rmsle = np.sqrt(metrics.mean_squared_log_error(y_valid, pred))
rmsle

np.float64(0.8874596405696397)